In [1]:
#import os
#os.environ["TRANSFORMERS_NO_TF"]='1'

In [ ]:
!pip uninstall tensorflow keras tf-keras-y

Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Would remove:
    /home/vanadzor-npua/jupyter/venv/bin/import_pb_to_tensorboard
    /home/vanadzor-npua/jupyter/venv/bin/saved_model_cli
    /home/vanadzor-npua/jupyter/venv/bin/tensorboard
    /home/vanadzor-npua/jupyter/venv/bin/tf_upgrade_v2
    /home/vanadzor-npua/jupyter/venv/bin/tflite_convert
    /home/vanadzor-npua/jupyter/venv/bin/toco
    /home/vanadzor-npua/jupyter/venv/lib/python3.12/site-packages/tensorflow-2.20.0.dist-info/*
    /home/vanadzor-npua/jupyter/venv/lib/python3.12/site-packages/tensorflow/*
Proceed (Y/n)? 

In [ ]:
!pip install python-telegram-bot==20.7 diffusers transformers torch torchvision torchaudio safetensors accelerate pillow nest_asyncio --upgrade --quiet

In [ ]:
pip install tf-keras

In [ ]:
# ===============================
# IMPORTS
# ===============================
"""import torch
from PIL import Image
from io import BytesIO
from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, MessageHandler, filters, ContextTypes
from transformers import pipeline
from diffusers import StableDiffusionImg2ImgPipeline
import asyncio

# ===============================
# DEVICE
# ===============================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ===============================
# TEXT GENERATION (English)
# ===============================
text_generator = pipeline(
    "text-generation",
    model="gpt2",
    device=0 if device == "cuda" else -1
)

# ===============================
# IMAGE MODIFICATION (img2img)
# ===============================
image_pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)
if device == "cuda":
    image_pipe = image_pipe.to("cuda")

# ===============================
# TELEGRAM HANDLERS
# ===============================
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text("Send me a photo and I will suggest a socially useful project for it!")

async def handle_photo(update: Update, context: ContextTypes.DEFAULT_TYPE):
    # Получаем фото
    photo_file = await update.message.photo[-1].get_file()
    photo_bytes = await photo_file.download_as_bytearray()
    init_image = Image.open(BytesIO(photo_bytes)).convert("RGB")

    await update.message.reply_text("Generating image and idea, please wait... ⏳")

    # ===============================
    # IMG2IMG
    # ===============================
    prompt = "A renovated and socially useful building, realistic architectural render"
    result = image_pipe(
        prompt=prompt,
        image=init_image,
        strength=0.35,
        guidance_scale=8.5,
        num_inference_steps=60
    )

    # Сохраняем результат
    new_image = result.images[0]
    bio = BytesIO()
    bio.name = "result.png"
    new_image.save(bio, "PNG")
    bio.seek(0)

    # ===============================
    # TEXT GENERATION (IDEA)
    # ===============================
    idea_prompt = (
        "You are a helpful assistant. Suggest one concrete and socially useful project "
        "for this building, in 1-2 sentences, in English."
    )
    idea = text_generator(
        idea_prompt,
        max_new_tokens=80,
        temperature=0.7,
        do_sample=True
    )[0]['generated_text']

    # Отправляем результат
    await update.message.reply_photo(photo=bio, caption=idea)

async def handle_text(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text("Please send a photo, not text, for project generation.")

# ===============================
# BOT SETUP
# ===============================
TOKEN = "8422134911:AAHbs1Wc08-fO7eoc80nav_XkAuL_SVPxnE"

app = ApplicationBuilder().token(TOKEN).build()
app.add_handler(CommandHandler("start", start))
app.add_handler(MessageHandler(filters.PHOTO, handle_photo))
app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_text))

# ===============================
# RUN BOT IN JUPYTER
# ===============================
async def run_bot():
    print("Бот запущен в Jupyter!")
    await app.start()
    await app.updater.start_polling()
    # Бот будет работать до остановки ядра
    await asyncio.Event().wait()

# Запускаем бота
asyncio.create_task(run_bot())

In [ ]:
#Anna
'''import os
import torch
import random
import asyncio
import nest_asyncio
from PIL import Image
from io import BytesIO
from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, MessageHandler, filters, ContextTypes
from diffusers import StableDiffusionImg2ImgPipeline

# 1. Jupyter-ի համար
nest_asyncio.apply()

# 2. Մոդելի բեռնում
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print(f"Մոդելը բեռնվում է {device} վրա...")
model_id = "runwayml/stable-diffusion-v1-5"
image_pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    model_id, 
    torch_dtype=dtype
).to(device)

# 3. Գաղափարների բազա
PROJECT_IDEAS = [
    {"label": "Ժամանակակից Դպրոց", "prompt": "modern futuristic school building, large windows, playground"},
    {"label": "Մանկապարտեզ", "prompt": "eco-friendly modern kindergarten, wooden facade, garden"},
    {"label": "Տեխնոլոգիական Կենտրոն", "prompt": "high-tech innovation hub, glass facade, futuristic office"},
    {"label": "Համայնքային Այգի", "prompt": "public community park, modern glass cafe, wooden benches"}
]

# 4. Հիմնական ֆունկցիա
async def handle_photo(update: Update, context: ContextTypes.DEFAULT_TYPE):
    try:
        # Ստանում ենք օգտատիրոջ նկարագրությունը (caption)
        user_text = update.message.caption if update.message.caption else ""
        
        if not user_text:
            await update.message.reply_text("💡 **Հուշում:** Նկարը ուղարկելիս `caption` դաշտում գրիր տեղանքի նկարագիրը (օր.՝ փողոցի մոտ է, շուրջը տներ կան)։")
            user_context = "urban area" # default context
        else:
            user_context = user_text

        await update.message.reply_text("🎨 Վերլուծում եմ տեղանքը և համադրում իմ գաղափարի հետ... ⏳")

        # Նկարի ներբեռնում
        photo_file = await update.message.photo[-1].get_file()
        photo_bytes = await photo_file.download_as_bytearray()
        init_image = Image.open(BytesIO(photo_bytes)).convert("RGB").resize((512, 512))

        # Գաղափարի ընտրություն և Prompt-ի միավորում
        selected = random.choice(PROJECT_IDEAS)
        
        # Այստեղ միացնում ենք քո նկարագիրը և AI-ի միտքը
        combined_prompt = (
            f"Architectural render of {selected['prompt']}, "
            f"located in a {user_context}, "
            f"high quality, realistic, cinematic lighting, 8k, detailed surroundings"
        )

        # Գեներացիա
        result = image_pipe(
            prompt=combined_prompt,
            image=init_image,
            strength=0.7, # 0.7-ը լավ բալանս է օրիգինալի և նորի միջև
            guidance_scale=9.0,
            num_inference_steps=40
        ).images[0]

        # Ուղարկում
        bio = BytesIO()
        bio.name = "project.png"
        result.save(bio, "PNG")
        bio.seek(0)

        response_text = (
            f"✅ **Տեղանքի նկարագիր:** {user_context}\n"
            f"💡 **AI-ի առաջարկ:** {selected['label']}\n\n"
            f"✨ Ես նախագծեցի {selected['label'].lower()}, որը ներդաշնակորեն համապատասխանում է քո նշված միջավայրին։"
        )
        
        await update.message.reply_photo(photo=bio, caption=response_text, parse_mode="Markdown")

    except Exception as e:
        await update.message.reply_text(f"Սխալ տեղի ունեցավ: {str(e)}")

# ... (Start և Run_bot ֆունկցիաները մնում են նույնը)

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "Ողջույն! Ուղարկիր նկար և նկարագրության մեջ գրիր, թե ինչ կա շուրջը (օրինակ՝ 'փողոցի եզրին, բնակելի տների մեջ'):\n\n"
        "Ես կհամադրեմ քո տեղեկությունը իմ գաղափարների հետ: 🏗️"
    )

TOKEN = "8422134911:AAHbs1Wc08-fO7eoc80nav_XkAuL_SVPxnE"
app = ApplicationBuilder().token(TOKEN).build()
app.add_handler(CommandHandler("start", start))
app.add_handler(MessageHandler(filters.PHOTO, handle_photo))

async def run_bot():
    print("Բոտը միացված է...")
    await app.initialize()
    await app.start()
    await app.updater.start_polling()
    await asyncio.Event().wait()

if __name__ == "__main__":
    asyncio.run(run_bot())'''

In [ ]:
import os
os.environ["TRANSFORMERS_NO_TF"]='1'

# Գրադարանների տեղադրում (անհրաժեշտության դեպքում)
# !pip install python-telegram-bot==20.7 diffusers transformers torch torchvision torchaudio safetensors accelerate pillow nest_asyncio --upgrade --quiet

import torch
import asyncio
from PIL import Image
from io import BytesIO
from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, MessageHandler, filters, ContextTypes
from diffusers import StableDiffusionImg2ImgPipeline

# ===============================
# DEVICE & MODEL SETUP
# ===============================
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print(f"Working on device: {device}")

model_id = "runwayml/stable-diffusion-v1-5"
image_pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    model_id, 
    torch_dtype=dtype
).to(device)

# ===============================
# FULL KEYWORDS DATABASE
# ===============================
keywords_map = {
    # Location & Vibe
    "busy": "busy grocery store with high foot traffic",
    "roadside": "modern boutique shop on a main road",
    "corner": "prominent corner cafe with glass windows",
    "downtown": "upscale office building in the city center",
    "quiet": "peaceful yoga studio or library",
    "residential": "neighborhood laundry service or small clinic",
    "noisy": "vibrant nightclub or urban music pub",
    "suburbs": "family-friendly pizzeria",
    "isolated": "minimalist smart home retreat",
    "touristic": "classic souvenir gift shop",
    "highway": "convenient gas station or motel",
    "bus stop nearby": "express pharmacy or newsstand",
    "near metro": "fast food outlet with neon signs",
    "park side": "ice cream parlor with outdoor seating",
    "bright": "sunlit flower shop",
    "dark": "moody underground wine bar",
    "basement": "hidden jazz club or speakeasy",
    "damp": "industrial mushroom farm or cellar",
    "windy": "modern observation deck with glass railings",
    "hilltop": "panoramic restaurant with a view",
    "riverside": "scenic tea house by the water",
    "lakeside": "luxury boutique hotel",
    "architectural": "modern art museum",
    "historic": "authentic antique shop",
    "modern": "high-tech coworking space",
    "ruined": "urban skate park or graffiti gallery",
    "half-destroyed": "industrial loft art gallery",
    "dusty": "vintage second-hand bookstore",
    "overgrown": "lush green garden center",
    "closed": "modernized community center",
    "abandoned": "creative startup hub",

    # Physical Attributes
    "stone": "traditional stone tavern with rustic finish",
    "brick": "trendy industrial brick barbershop",
    "glassy": "high-tech glass laboratory",
    "metallic": "modern metalwork atelier",
    "wooden": "eco-friendly wooden cottage cafe",
    "high ceiling": "spacious loft-style gym",
    "low": "cozy small bar or cobbler shop",
    "one-story": "charming artisan bakery",
    "multi-story": "modern urban hostel",
    "circular": "futuristic cinema or planetarium",
    "square": "organized local supermarket",
    "narrow": "minimalist coffee-to-go kiosk",
    "spacious": "grand exhibition hall",
    "large windows": "bright beauty salon",
    "windowless": "professional recording studio",
    "arches": "classic wine cellar with arched ceilings",
    "columns": "prestigious school or government building",
    "balcony": "romantic rooftop cafe with a balcony",
    "rooftop": "vibrant rooftop lounge bar",
    "entrance": "welcoming information center",
    "canopy": "outdoor flower market with a canopy",
    "rusty": "gritty industrial-style pub",
    "colorful": "playful kids development center",
    "grey": "minimalist professional office",
    "white": "clean dental clinic or medical spa",
    "black": "luxury watch or automotive showroom",
    "dirty": "alternative underground art space",
    "renovated": "sleek modern business center",
    "shiny": "polished electronics flagship store",
    "smooth": "contemporary art gallery with smooth surfaces"
}

# ===============================
# LOGIC FUNCTIONS
# ===============================
def create_ai_prompt(user_text):
    if not user_text:
        return "modern architectural visualization of a renovated building, 8k, photorealistic"
    
    user_text = user_text.lower()
    found_matches = []

    # Check for keywords in user input
    for key, prompt_val in keywords_map.items():
        if key in user_text:
            found_matches.append(prompt_val)

    if not found_matches:
        # If no keywords found, use user input directly as a prompt
        return f"Architectural transformation of {user_text}, hyper-realistic, 8k, professional lighting"
    
    # Combine all matched prompt segments
    combined_ideas = ", ".join(found_matches)
    return f"Professional architectural visualization, an old building transformed into {combined_ideas}, photorealistic, highly detailed exterior, 8k resolution"

# ===============================
# TELEGRAM HANDLERS
# ===============================
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "Welcome! 🏗️\nSend me a photo of an old building and write a description in the caption.\n"
        "Example: 'a busy street corner with stone walls'."
    )

async def handle_photo(update: Update, context: ContextTypes.DEFAULT_TYPE):
    try:
        user_caption = update.message.caption or ""
        
        # 1. Download Photo
        photo_file = await update.message.photo[-1].get_file()
        photo_bytes = await photo_file.download_as_bytearray()
        init_image = Image.open(BytesIO(photo_bytes)).convert("RGB")
        init_image = init_image.resize((512, 512))

        await update.message.reply_text("Analyzing keywords and generating your project... ⏳")

        # 2. Generate Prompt based on the Map
        final_prompt = create_ai_prompt(user_caption)
        print(f"Final Prompt: {final_prompt}")

        # 3. AI Generation (Img2Img)
        # Using strength 0.75 to balance original structure and new design
        result = image_pipe(
            prompt=final_prompt,
            image=init_image,
            strength=0.75, 
            guidance_scale=9.0,
            num_inference_steps=40
        ).images[0]

        # 4. Save to buffer
        bio = BytesIO()
        bio.name = "renovation.png"
        result.save(bio, "PNG")
        bio.seek(0)

        # 5. Send result
        caption_text = f"✨ **New Project Idea**\n\n🔍 **Detected Style:** {user_caption if user_caption else 'Default Modern'}"
        await update.message.reply_photo(photo=bio, caption=caption_text, parse_mode="Markdown")

    except Exception as e:
        await update.message.reply_text(f"Error: {str(e)}")

# ===============================
# RUN BOT
# ===============================
TOKEN = "8422134911:AAHbs1Wc08-fO7eoc80nav_XkAuL_SVPxnE"

app = ApplicationBuilder().token(TOKEN).build()
app.add_handler(CommandHandler("start", start))
app.add_handler(MessageHandler(filters.PHOTO, handle_photo))

if __name__ == "__main__":
    import nest_asyncio
    nest_asyncio.apply()
    
    print("Bot is running...")
    app.run_polling()